In [ ]:
------------------------------------------------------------------------------------------------------------
DataScienceCourse/
├── notebooks/
│   ├── 00_template.ipynb           # Template, for copying and having relative paths functional.
│   ├── 01_limpieza_grupo1.ipynb
│   └── 05_eda_analisis.ipynb
├── src/
│   ├── __init__.py
│   ├── paths.py                    # Pathing Setup script
│   └── descriptive_tools.py        # Script datascience ds3on1, vartype, statsgroup
└── main.py                         # Entry point for the scripts./
        "ROOT": ROOT_DIR,
        "DATA": DATA_DIR,
        "TABLES": TABLES_DIR,
        "PROCESSED": PROCESSED_DATA_DIR
------------------------------------------------------------------------------------------------------------

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import os

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src import setup_workspace, query_to_df, ds3on1

# 1. Inicializar workspace y rutas
paths = setup_workspace()

# 2. Consultar directamente a PostgreSQL (vía DuckDB ATTACH)
#df = query_to_df("SELECT * FROM db.EuInfo;")

# 3. Análisis univariable 3-en-1
#ds3on1(df, 'happy', Desc=True, Stat=True, Graph=True)

🟢 < Module 7.1 > Exploratory Data Analysis (EDA), Stratified Distributions & Correlation Diagnostics

##### 🎯 Context & Objectives

In Exploratory Data Analysis (EDA), the goal is not merely to compute global summary statistics, but to uncover structural patterns, cluster behaviors, and dependencies between predictors and target variables across different economic contexts. In this exercise, you will perform an advanced EDA on your cleaned master survey dataset (`surveys_clean` combined with macro features). You will analyze ordinal vs. continuous variable distributions, evaluate non-linear relationships, and construct stratified correlation heatmaps comparing different European country clusters (e.g., high-income vs. medium/low-income economic groups).

##### 📑 Exercise Instructions

###### 1. Syntactic / Algorithmic Level (Pandas & Seaborn EDA Fundamentals)

Create a Python notebook (`notebooks/07_1_exploratory_data_analysis.ipynb`) that implements the following technical steps:

- **Statistical Profiling & Distribution Metrics** :
    
    - Compute robust central tendency and dispersion metrics (median, IQR, mean, standard deviation) for key survey ordinals (e.g., `happy`, `lrscale`, `stfdem`) and engineered features (`EmployType`, `emplno`).
        
    - Identify skewness and modal concentrations (such as the point mass at `0` in `emplno` for wage earners).
        
- **Stratified Correlation Analysis (Spearman vs. Pearson)** :
    
    - Explain why **Spearman’s rank correlation** is theoretically superior to Pearson's correlation when analyzing ordinal Likert scales (0–10) in survey data.
        
    - Generate a correlation matrix and visualize it using a Seaborn heatmap (`sns.heatmap`), ensuring proper diverging colormaps and annotation of correlation coefficients.
        

###### 2. Applied / Data Level (Socio-Economic Stratification & Insights)

Evaluate how macroeconomic country clusters modulate individual behaviors:

- **Multi-Group Comparison** :
    
    - Group your dataset by country economic tiers or regional clusters (`cntgrp_fc` / `std_region`).
        
    - Compare the distribution of political ideology (`lrscale`) or institutional trust (`trstlgl`) across different economic clusters using boxplots or violin plots (`sns.boxplot` / `sns.violinplot`).
        
- **Insight Synthesis** :
    
    - Document 3 key empirical findings from your visualizations that support your core research hypothesis (e.g., how educational level `edulvlb` or employment relation `EmployType` shifts political or social trust distributions).
        

##### 🌐 Official References & Documentation

- **Seaborn Statistical Plots** : [Seaborn Heatmap & Distribution Tutorial](https://seaborn.pydata.org/generated/seaborn.heatmap.html)
    
- **Pandas Statistical and Descriptive Functions** : [Pandas Descriptive Statistics Guide](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
    

##### 📚 Recommended Manuals & CheatSheets

- **DataCamp Seaborn Cheat Sheet** : [Python Seaborn Data Visualization Cheat Sheet](https://www.datacamp.com/cheat-sheet/python-seaborn-cheat-sheet)
    
- **Exploratory Data Analysis Best Practices** : [Python for Data Analysis — Exploratory Data Analysis Chapter](https://wesmckinney.com/book/)
    

💻 **Your Challenge:** Create the notebook `notebooks/07_1_exploratory_data_analysis.ipynb` in your local environment, run the stratified descriptive statistics and correlation heatmaps, and share your code or key findings here when ready for review! 🚀📊

In [ ]:
#First of all lets load a sample dataframe for this excercise from the main Surveys table.
import duckdb as db
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

load_dotenv()

pg_db = os.getenv("PGDATABASE")
pg_user = os.getenv("PGUSER")
pg_pass = os.getenv("PGPASSWORD")
pg_host = os.getenv("PGHOST")
pg_port = os.getenv("PGPORT")


con = db.connect()
    

con.execute("INSTALL postgres; LOAD postgres;")
    

attach_query = f"""
ATTACH IF NOT EXISTS 'dbname={pg_db} user={pg_user} password={pg_pass} host={pg_host} port={pg_port}' 
AS db (TYPE postgres);
"""
    

con.execute(attach_query)


df_raw = con.execute("""
    SELECT emplno, agea, emplrel, edulvlb, happy, lrscale, cntry, EmployType, trstprl, trstlgl
    FROM db.surveys
    WHERE COLUMNS(*) IS NOT NULL
""").df()

df_71 = pd.concat([
    g.sample(n=min(100, len(g)), random_state=42)
    for _, g in df_raw.groupby('cntry')
]).reset_index(drop=True)

print(f"Shape: {df_71.shape}")  # → (2400, 8)
del df_raw 

In [ ]:
------------------------------------------------------------------------------------------------------------
Step 0, Problem framing:
-Research question:
 To what extent do regional location, national political-economic context, and country-specific factors explain variation in [variable social var] 
across European countries, 2004–2024?
-RQ1 (descriptive): How do [surveys_vars] vary across European regions and over time (2004–2024)?
-RQ2 (comparative): Is the variance in [variable social] between countries larger than the variance between regions within the same country?
-RQ3 (asociative): To what extent do national economic indicators (GDP per capita, unemployment, Gini) predict [surveys_vars] within and across countries?
-RQ4: Do European countries cluster more strongly by similarity in economic trend trajectories (2004–2024) than by national identity (political regime, 
cultural region, or historical bloc)?
------------------------------------------------------------------------------------------------------------

In [ ]:
#First of all lets do a quick univariate analysis.
#1)Univariate Data Inspection:

ds3on1(df_71, 'emplno', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
-emplno: – Number of employees the respondent has (self-employed only).  Discrete count, non-negative.
        -The distribution is zero-inflated: ~96% of respondents report 0 employees, reflecting that the vast majority are salaried workers, 
own-account self-employed, or not in the labour force. The non-zero values (1, 2, 5, 10, 25, 50+) represent a small subset of employers 
and are right-skewed with high kurtosis value(Leptokurtic. Almost all the proportion is located on 0 value).
        -Implication for analysis: this variable cannot be treated as a standard numeric variable. Options:
        -Create a binary indicator (is_employer: 0/1) for the RQs that care about whether someone employs.
        -For the subset of employers only, model the count separately (log-transform, or zero-inflated/negative binomial if modelling).
        -Do not use the raw count in a regression with the full sample — the 96% zeros will dominate and mask any signal.
------------------------------------------------------------------------------------------------------------

In [ ]:
ds3on1(df_71, 'agea', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
-agea: – Age of the respondent. Discrete, non-negative integer.
        -Outliers / sentinel values: 666, 777, 888, 999 are non-response or data-entry error codes, not valid ages. They severely inflate the mean 
and must be excluded before any summary statistic is computed.
        -Distribution (after removing sentinels): Approximately symmetric, slightly right-skewed, with a median of ~53 years. 
The IQR spans 37–67, meaning half the sample is concentrated in a 30-year window. The tails under-represent young adults (18–36) and the oldest cohort (68+).
        -Implication: Age is a potential confounder for [variables sociales]. The concentration in 37–67 limits generalizability to younger 
and older populations. Any model that includes age as a covariate should use the median or a binned/categorical version rather than the raw (sentinel-inflated) value.
        -Cleaning decision: Remove sentinel codes (666, 777, 888, 999) and values < 18 (below survey eligibility per [source]).
------------------------------------------------------------------------------------------------------------

In [ ]:
ds3on1(df_71, 'emplrel', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
-emplrel: – Attends on wich type of job the respondent has. A factorized categorical variable, where 1 is all the respondents are wage employe and 2 self-employers or bussinessmen
          -The distribution is one inflated proportion, almost 90% of respondents are wage employers.
          -This variable it's totally derivated from emplno, and almost has the same distribution.
          -We can do the same method as 'emplno' or just omit it in the feature engineering.
------------------------------------------------------------------------------------------------------------

In [ ]:
ds3on1(df_71, 'edulvlb', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
-edulvlb: – Education and profesional level of the respondent, ISCED codification . Discrete, non-negative ordinal value (the more qualification, higher code respondent has.).
          - Outliers / sentinel values: 6666, 7777, 8888, 9999 are non-response or data-entry error codes. Also there is a respondent minority class on 0 value, 
"ISCED 0 (early childhood / no formal education): ~1% of respondents"
          - Median observation tell us our frecuency or porportion is located with mayor mass in the Q2-Q3 range. We can observe an non high leptokurtosis and right-skewed distribution.          
          - The 50% of our mass fluctuates from 313(Q1)ISCED code to 610(Q3) code. This mas is located in 297(IQR) codification range.
          - This variable could be transformed in 3 or 4 categorical groups, also be binned in 3 variables in the case we need for the feature engineering.
------------------------------------------------------------------------------------------------------------                                                                  

In [ ]:
ds3on1(df_71, 'happy', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
-happy: – State of happyness the respondent has, being 0 unhappy and 10 the max value on lickert scale. Discrete, non-negative ordinal value. 15 ordinal unique values
        - Outliers / sentinel values: 66, 77, 88, 99 are non-response or data-entry error codes.
        - The mean (7.8) is slightly below the median (8), consistent with the left skew: the thin tail of low-satisfaction responses (0–5) pulls the mean downward, while the median remains anchored in the central mass at 7–8.
        - The iqr range is so compressed, and the mean-median relation tells us in that 50% of mass respondents, respondents are more close to 8 value of happy than 7.
        - Implication: The limited dynamic range in the central mass makes the raw variable poorly discriminative for modelling.
        Options:
            -Binarize: satisfied = happy ≥ 8 (aligns with Eurostat's own "satisfied / not satisfied" split). Best if the RQ is "what predicts being satisfied?"
            -Categorical (3–4 groups): Unhappy (0–4), Neutral (5–6), Satisfied (7–8), Very satisfied (9–10). Best for ordinal regression (proportional odds).
            -Keep as interval (0–10): Defensible for OLS / random effects if you assume equal spacing. Best if the RQ is "what predicts the level of satisfaction?" and you want to use the full range.
        The choice depends on which RQ it serves. For RQ3 (economic predictors), the interval version preserves more information; for RQ4 (clustering), the categorical version is more robust.
------------------------------------------------------------------------------------------------------------

In [ ]:

col=df_71['cntry']
def quick_stats(col):
    mode = col.mode()
    return pd.Series({
        'count':  col.count(),
        'categories count': col.nunique(),
        'Uniques frecuency': col.value_counts(),
        #'mean':   col.mean(),
        #'std':    col.std(),
        'min':    col.min(),
        #'Q1':     col.quantile(.25),
        #'median': col.median(),
        #'Q3':     col.quantile(.75),
        'max':    col.max(),
        #'mode':   mode[0] if not mode.empty else np.nan,
        #'skew':   col.skew(),
        #'kurtosis': col.kurt()
    }) 
print(quick_stats(col))
#Being non factorized discrete categorical column, most of our metrics can't be used.
import matplotlib.pyplot as plt
import seaborn as sns
# 1. Sort and calculate frecuencies
counts = df_71['cntry'].value_counts().reset_index()
counts.columns = ['cntry', 'count']

plt.figure(figsize=(8, 6))
# 2. Swap the axis being Y axis the country codes and X the number of rows.
sns.barplot(data=counts, x='count', y='cntry', palette='viridis', legend=False)

plt.title('Survey Countries')
plt.xlabel('Quantity of people from each country')
plt.ylabel('Country')
plt.show()

In [ ]:
------------------------------------------------------------------------------------------------------------
-cntry: – the string values (countries) are ordered in order of appearance, 
it is the variable that we have used to create the dataframe using grouping and maximum number of random samples.
------------------------------------------------------------------------------------------------------------

In [ ]:
ds3on1(df_71, 'lrscale', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
-lrscale: – Lickert Scale wich gives us information of the political ideology respondent was, 0 is comunism, and 10 far right extremism. Discrete, non-negative ordinal value. 15 ordinal unique values
          - Outliers / sentinel values: 66, 77, 88, 99 are non-response or data-entry error codes.
          - The mean is highly affected by positive outliers of non respondent values.
          - The median tells us that Q2 finishes and splits the graph at 5 value. The 50% of mass proportion its located between 4 and 7 (Q1/Q3), fluctuating on 3 discrete values(IQR).
The iqr tells us that those interviewed have a fairly standard political position, from social democracy to a non-conservative right-wing position.
          - In this case the shape is right-skewed, also  leptokurted on iqr range.
          Options:
            - Same case as 'Happy', 1- binarize left wing- Right wing, 2 - categorize in more groups to detect more clases for better for ordinal regression.
            - Make it continuous in the case we focus the problem as range prediction.
            - Keep this ordinal interval, assuming the inequality of proportion.
------------------------------------------------------------------------------------------------------------

In [ ]:
ds3on1(df_71, 'EmployType', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
EmployType: We can discard this variable on EDA, it has same distribution and mass proportion than Emprel.
------------------------------------------------------------------------------------------------------------

In [ ]:
ds3on1(df_71, 'trstprl', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
trstprl - Trust in country's ' parliament, Discrete, positive, ordinal lickert scale from 0 to 10, with non respondent values and non valid from 66 to 99. 15 ordinal unique values
        - We can observe in the distribution and shape the mass is very distributed on its range(slightly right skewed, and low kurtosis(the mass is balanced arround a wide range of variable values)), also the distribution is contaminated with the outliers that attends on non valid values (66,77,88,99), altering the shape and mean value. 
The 0 class has almost 12% of frecuency.
        - The 50% of respondent mass are distribuited in the almost half of variable range"4.25"/IQR, from 2.75/Q1 to 7/Q2
        - This column is very balanced, and also could be a good target variable. We can keep it as ordinal, binnarize or make categorical grouping.
        Options: We can binarize the column, aswell make categorical grouping or and especific continuous transformation.
------------------------------------------------------------------------------------------------------------

In [ ]:
 ds3on1(df_71, 'trstlgl', Desc=True, Stat=True, Graph=True)

In [ ]:
------------------------------------------------------------------------------------------------------------
trstlgl - Trust in legal system. Discrete, positive, ordinal lickert scale, from 0 to 10, with non respondent/valid values from 66  to 99
        - Again the right skewed of the shape indicates a data contamination due non respondent/valid values located on 66-99 column values, 
we can observe aswell a bit kurtosis around the median, our mean is displaced due positive outliers density.
        - The 4/IQR range indicates a decent spread mass of the 50% of samples. almost centered in a logic response, low untrust "4"/Q1 and decent trust "8"/Q3
        Options: We can binarize the column, aswell make categorical grouping or and especific continuous transformation.
------------------------------------------------------------------------------------------------------------

In [ ]:
------------------------------------------------------------------------------------------------------------
2) Multivariate analysis — data preparation

A) Before proceeding, we clean the dataset of non-respondent codes (66, 77, 88, 99 for ordinal scales; 
6666, 7777, 8888, 9999 for continuous variables) detected per column. Each occurrence is recorded in a new integer column n_nonresp, 
counting how many non-valid responses each respondent gave across the trust variables.

B) The resulting missing values are then imputed via random hot-deck: each missing value is drawn from the empirical distribution of valid responses for that column, 
preserving the original shape, variance, and discrete support of the scale.

C) The final preparation step consists of stratifying the sample along three axes: socioeconomic (income, employment status), 
regional (autonomous community / NUTS-2), and response completeness (n_nonresp tiers). For each stratum, we examine how the distributional shape of 
trust variables shifts — comparing medians, IQRs, and mass concentration across groups.

------------------------------------------------------------------------------------------------------------

In [ ]:
# 2.1) Data Preparation.
#A) Value filtering function, this function searches non valid values, starting from mayor numbers, clustered on 4 posible numbers in 4 different lists. Stores this information and then inputs nan on those value cells.
# Also makes a control variable called "n_nonresp" whenever there is an invalid or unanswered value in the row, the n_nonresp counter adds one
def clean_and_count(df):

    df = df.copy()
    df["n_nonresp"] = 0
    report = {}

    # Grupos de mayor a menor cifra
    CODE_GROUPS = [
        [99999, 88888, 77777, 66666],  # 5 dígitos
        [9999,  8888,  7777,  6666],   # 4 dígitos
        [999,   888,   777,   666],    # 3 dígitos
        [99,    88,    77,    66],     # 2 dígitos
    ]

    for col in df.columns:
        if col == "n_nonresp":
            continue

        unique = set(df[col].dropna().unique())

        # Embudo: primero el grupo de mayor cifra
        codes_found = None
        for group in CODE_GROUPS:
            hit = [c for c in group if c in unique]
            if hit:
                codes_found = hit
                break  # se cierra con este grupo, no se mira el siguiente

        if codes_found is None:
            continue  # ningún código presente, se descarta

        # Detección
        mask_invalid = df[col].isin(codes_found)
        n_invalid = int(mask_invalid.sum())

        # Contador
        df.loc[mask_invalid, "n_nonresp"] += 1

        # Setear a NaN
        df.loc[mask_invalid, col] = np.nan

        report[col] = {
            "codes": codes_found,
            "n_invalid": n_invalid,
            "pct_invalid": round(n_invalid / len(df) * 100, 2)
        }

    return df, report   

In [ ]:
#A) Appliyng the function:
df_71, report = clean_and_count(df_71)  # ← reasignas   

In [ ]:
df_71['n_nonresp'].describe()

In [ ]:
#2.1.B) Hot-deck random imputation, we will consider filling NAN values with a weigthed value, attending on original density/distribution of each column.

def hot_deck_impute(df, seed=42):

    rng = np.random.default_rng(seed)
    df = df.copy()
    report = {}

    for col in df.columns:
        if col == "n_nonresp":
            continue

        n_nan = df[col].isna().sum()
        if n_nan == 0:
            continue

        # Distribución empírica de valores válidos
        valid = df[col].dropna()
        weights = valid.value_counts(normalize=True)

        # Muestreo con reemplazo
        imputed = rng.choice(weights.index, size=n_nan, p=weights.values)

        df.loc[df[col].isna(), col] = imputed

        report[col] = {
            "n_imputed": int(n_nan),
            "n_valid_before": int(len(valid)),
            "pct_imputed": round(n_nan / len(df) * 100, 2)
        }

    return df, report 

In [ ]:
df_71, imp_report = hot_deck_impute(df_71, seed=42)
print(imp_report)
 

In [ ]:
df_71.info()

In [ ]:
#Before multivariate visualization and clustering.
#Lets transform all ordinals in to int again.
def cast_to_int(df, cols):
    
    df = df.copy()
    for col in cols:
        df[col] = df[col].astype(int)
    return df 


In [ ]:
df_71 = cast_to_int(df_71, ["agea", "edulvlb", "happy", "lrscale", "trstprl", "trstlgl"])
df_71.info()

In [ ]:
#Grouping by logic clusters (economic, macro region, n_nonresp value and non clustered)
#Lets use our Europe information table created in the branch "git checkout Datacollect1", with macro region information and eurostat-wordlbank data.

con.execute(attach_query)
df_Eu = con.execute("SELECT * FROM db.macro_panel").df()
df_Eu.info()





In [ ]:
#Lets get only the values for the 2024 year.
df_macro_2024 = df_Eu[df_Eu["year"] == 2024].copy()
print(df_macro_2024.shape)  # debería ser (15, 8)   

In [ ]:
print(df_Eu["year"].unique())  # Registered years data
print(df_macro_2024.shape)     # how many columns
print(df_macro_2024.columns.tolist())  # ¿the name of columns

In [ ]:
#Our Economic information is limited due the lack of information of some of newest Eu regions,
# We will delete from our df_71 those countries without economic data, because we want to see
# how our data distribution is affected when grouping or non grouping.
# Countries with data information.
valid_countries = set(df_macro_2024["geo"].unique())
macro_cols = ['macro_region','gdpcapitapct','ginicoefpct','inflationpct', 'reglangdemographypct','unemploymentpct']
                                                                                                 
# filter df_71
df_71_clean = df_71[df_71["cntry"].isin(valid_countries)].copy()
print(f"Filas antes: {len(df_71)} → después: {len(df_71_clean)}")
print(f"Países: {df_71_clean['cntry'].nunique()}")

# Clean merge
df_71_clean = df_71_clean.merge(
    df_macro_2024[["geo"] + macro_cols],
    left_on="cntry", right_on="geo", how="left"
)
print(df_71_clean["gdpcapitapct"].isna().sum())  # 0   

In [ ]:
df_71.info()

In [ ]:
del df_71, df_macro_2024, df_Eu

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler   
def kmeans_economic(df, macro_num_cols, n_clusters=3, seed=42):

    macro_df = df[["cntry", "macro_region"] + macro_num_cols].drop_duplicates(subset="cntry").reset_index(drop=True)

    neg_vars = ['ginicoefpct', 'inflationpct', 'unemploymentpct']
    pos_vars = [c for c in macro_num_cols if c not in neg_vars]

    for col in neg_vars:
        macro_df[f"{col}_inv"] = macro_df[col].max() - macro_df[col]

    cluster_cols = pos_vars + [f"{c}_inv" for c in neg_vars]

    X = macro_df[cluster_cols].values
    X_scaled = StandardScaler().fit_transform(X)

    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    macro_df["econ_cluster"] = kmeans.fit_predict(X_scaled)

    # Etiquetar por GDP medio: 0=bajo, 1=medio, 2=alto
    gdp_by_cluster = macro_df.groupby("econ_cluster")["gdpcapitapct"].mean().sort_values()
    label_map = {old: new for new, old in enumerate(gdp_by_cluster.index)}
    macro_df["econ_cluster"] = macro_df["econ_cluster"].map(label_map)

    df = df.merge(macro_df[["cntry", "econ_cluster"]], on="cntry", how="left")

    return df, macro_df   

In [ ]:
macro_num_cols = ['gdpcapitapct', 'ginicoefpct', 'inflationpct', 'unemploymentpct']
df_71_clean, macro_detail = kmeans_economic(df_71_clean, macro_num_cols, n_clusters=3)   
print(macro_detail[["cntry", "gdpcapitapct", "unemploymentpct", "econ_cluster"]].to_string())   

In [ ]:
# The clustering reveals that GDP alone does not determine economic profile. Great Britain, despite a GDP comparable to Austria and a similar unemployment rate, is assigned to a lower cluster due to its Gini coefficient (35.1) 
# being the highest in the sample. This suggests the algorithm captures not just income generation but distributional quality as a differentiating axis."

# 3) Multivariate analysis (we will use cntry, macro_region, econ_cluster and n_nonresp to see how this groups alters distribution and shape from our variables)
#Lets define cluster cols for group the data when we compare the other kind of variables/columns.
#We will use general data(all countries), especific countries, in my case cntry==ES, and the rest of group cols.
df_71_clean['is_ES'] = (df_71_clean['cntry'] == 'ES').astype(int)
group_cols = ['is_ES', 'macro_region', 'econ_cluster', 'n_nonresp']
#Lets define the observational cols that will be grouped and compared each other.
study_cols= ['emplno','agea','emplrel','edulvlb','happy','lrscale','EmployType','trstprl','trstlgl']


In [ ]:
#Lets use a plot function for all our variables, all our categorical columns for inside of study_cols group are already factorized.
#And compare the distribution within different group_cols
def multivariate_plot(df, study_col, group_cols):


    for group_col in group_cols:
        groups = sorted(df[group_col].dropna().unique())
        n_groups = len(groups)

        fig, axes = plt.subplots(2, n_groups, figsize=(3 * n_groups, 7))
        if n_groups == 1:
            axes = axes[:, np.newaxis]  # asegurar 2D

        for j, g in enumerate(groups):
            subset = df[df[group_col] == g][study_col].dropna()

            # --- Fila 0: Bar plot ---
            ax_bar = axes[0, j]
            counts = subset.value_counts().sort_index()
            ax_bar.bar(counts.index.astype(str), counts.values, color='steelblue', edgecolor='white')
            ax_bar.set_title(f"{study_col} | {group_col}={g}\n(n={len(subset)})", fontsize=8)
            ax_bar.set_xlabel("")
            ax_bar.set_ylabel("Frecuencia")

            # --- Fila 1: Violin + box inner ---
            ax_violin = axes[1, j]
            sns.violinplot(y=subset, ax=ax_violin, inner='box',
                           color='steelblue', cut=0, bw_adjust=0.7)
            ax_violin.set_title(f"{study_col} | {group_col}={g}\n(n={len(subset)})", fontsize=8)
            ax_violin.set_xlabel("")
            ax_violin.set_ylabel("")

        plt.suptitle(f"{study_col} × {group_col}", fontsize=12, y=1.02)
        plt.tight_layout()
        plt.show()   

In [ ]:
multivariate_plot(df_71_clean, 'emplno', group_cols)   

In [ ]:
------------------------------------------------------------------------------------------------------------
'emplno' - We can observe in the first group, how changes de distribution of emplno, when we only observe only on Spanish
samles. IN our 100 samples we cant see any respondent with employes. That means the predominant class for Spanish respondents are
wage employers or ppl without work/students.
        - In this dataset having employers is a minority.
        - Surprisingly the group who offer more respondents with employes its northern and southern region.
        - From the economic view, Econ cluster == 1 (The most common countries in the EU, like Spain, France or Great Britain.)
are who has more respondents with employes aswell.
        -Non respondent view doesnt give any clear correlation betwen groups and distributions.
------------------------------------------------------------------------------------------------------------

In [ ]:
multivariate_plot(df_71_clean, 'agea', group_cols)  

In [ ]:
------------------------------------------------------------------------------------------------------------
'agea' - In the first group comparison (Spanish or not) we can observe how the distribution changes from unimodal shape
around 60 years old to bimodality with low kurtosis(the density is well spread) exactly in the age of 35 and 66 years old.
        - The second observation group (macro regions) we can observe different density nodes between the groups, easter has 3 density
nodes on 35, 50 and 65 years old having a good wide representation. Norther is the most balanced between these subgroups. Having a
bit more density on 35 and 45 years old. Southern describes a low kurtosis on 48 years old age, meanwhile western does it around 58 years
old.
We just have 100 samples for each country, so is too quick to evaluate the whole dataset.
        - The third group, economy clusters, the poor ones follows similar graph than eastern macro region, the normal economy countries
has a good representation, having a bit more density on 48 years old, meanwhile rich countries(econ_cluster==2) their density node is around
58 years old, at the same time, its IQR is the most wide between the economic subgroups.
        -In the non response observation subgroups we can observe, that on people with 0 responses its age has a density node on 58 years
old, with an iqr a bit concentrated than non respone =1 having the median higher than the subgroup 2 wich has a bimodality or 2 high
density nodes on 75 years old and 45 years old. Anyways i think hasnt to much correlation because the ages of non respondent doesnt 
have any pattern , it changes totaly when n_nonresp increases.
------------------------------------------------------------------------------------------------------------

In [ ]:
------------------------------------------------------------------------------------------------------------
Lets study directly lrscale, due its high relation with non respone values.
------------------------------------------------------------------------------------------------------------

In [ ]:
multivariate_plot(df_71_clean, 'lrscale', group_cols)

In [ ]:
------------------------------------------------------------------------------------------------------------
'lrscale' - In the first comparison group (Es=0/Es=1) we can observe how the denisity loses its mass on 5 value, spreading samples
in the whole range of the varible, its median is similar but iqr increases near to 3 unique value, means that spanish is more common to see
left winged respondents more than the common of the rest of Europe.
          - In the second groupd (macro regions) we can see how the distribution varies slightly in terms of density around the median,
there is a bit more of density on right wing respondents in Eastern countries, on Northern countries we see 2 extra density nodes on 3 and 8 lrscale values, 
in the southern macro region has a bit more density around 5 lrscale value in comparison with western values, wich has high density and kurtosis in 5 lrscale value.
          - In the third group (economic clusters) we can observe same pattern as macro regions, being the poors who got more representation on higher right wing ideology. 
The other 2 economic clusters follows a more limited density around 5 lrscale value, being the economic cluster 2 who has a greater kurtosis on this value.
          - In the non valid/responded group, we can observer , the higher non respond value the higher is the absence of lrscale values located near to 0 and 10.
------------------------------------------------------------------------------------------------------------          

#4) Multicomparison, correlation matrix:
#  Spearman's rank correlation is used throughout because the majority of variables are ordinal (0-10 Likert scales),
#  categorical (one-hot encoded), or binary. Spearman operates on ranks, making it the natural choice for all three types simultaneously. 
#  The few remaining numeric variables (agea, emplno) do not require transformation, as Spearman internally converts them to ranks, treating them as ordinal by default.

In [ ]:
# One-hot there is no natural order between them.
region_dummies = pd.get_dummies(df_71_clean['macro_region'], prefix='region')
df_71_clean = pd.concat([df_71_clean.drop(columns=region_dummies.columns, errors='ignore'), region_dummies], axis=1)     
# → region_eastern, region_northern, region_southern, region_western   
#We can one-hot econ cluster also, but it has already natural order being 0 poor and 2 rich, it can be usefull for the feature engineering.

# Correlaction dataframe
corr_cols = ['agea', 'happy', 'lrscale', 'trstprl', 'trstlgl',
             'gdpcapitapct', 'ginicoefpct', 'inflationpct',
             'unemploymentpct', 'n_nonresp', 'econ_cluster','region_eastern','region_northern', 'region_southern','region_western'] 

corr = df_71_clean[corr_cols].corr(method='spearman')

In [ ]:
corr.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).format('{:.3f}').set_properties(**{'text-align': 'center', 'font-size': '11px'})

In [ ]:
------------------------------------------------------------------------------------------------------------
Let's make a breaf analysis by semantic variable blocks:

- Trust block:
"Trust in parliament and trust in the legal system are strongly correlated (ρ = 0.63), forming a coherent institutional trust block. 
"Neither ideological position (ρ = 0.06), 
"subjective well-being (ρ = 0.05), nor age (ρ = −0.05) show meaningful association with institutional trust in this sample."

- Trust block - Macroeconomic Variables:
"Among macroeconomic variables, the Gini coefficient is the strongest predictor of institutional trust (ρ = −0.26 for parliament, −0.18 for legal system), 
"followed by GDP per capita (ρ = 0.19 / 0.23). Unemployment shows no meaningful association (ρ ≈ 0.03), suggesting that institutional trust is driven by 
"the level and distribution of wealth rather than labor market conditions."

- Trust block - Macro Region dummies:
"The northern region shows the highest association with institutional trust (ρ = 0.27), while southern, eastern, and western regions show negative or null associations.
"However, the northern region's correlation is nearly identical to the economic cluster's (ρ = 0.24–0.28), 
"indicating that the 'Nordic effect' is largely an economic effect rather than a distinct institutional or cultural one."

- Quality  Response block:
"Response completeness shows a weak negative association with trust in parliament (ρ = −0.13), 
"confirming that non-respondents are not a random subset: they tend to be less trusting. 
"This validates the need for the n_nonresp control variable in the predictive model."

- Macroeconomic (group and individual) block:
"
| Pair                             | r          | Interpretation              |
| -------------------------------- | ---------- | ----------------------------|
| `gdpcapitapct ↔ ginicoefpct`     | −0.349     | More PIB → less inequality. |
| `gdpcapitapct ↔ inflationpct`    | −0.257     | More PIB → less inflation.  |
| `inflationpct ↔ unemploymentpct` | **−0.397** | Phillips in cross section.  |
| `gdpcapitapct ↔ unemploymentpct` | −0.240     | More PIB → less unemployment|
"
- Macroeconomic cluster - Macro region (redundance):
"
|                                  |            |                |
| -------------------------------- | ---------- | -------------- |
| Pair                             | r          | Interpretation |
| `region_northern ↔ econ_cluster` | 0.290      | Moderate.      |
| `region_southern ↔ econ_cluster` | −0.302     | Moderate.      |
| `region_western ↔ econ_cluster`  | 0.395      | Moderate-high. |
| `region_eastern ↔ econ_cluster`  | **−0.655** | Strong.        |
"
- In this exercise the next feautures would be my set for the prediction model:
"
| Feature           | r con trstprl | Decission                                 |
| ----------------- | ------------- | ----------------------------------------- |
| `trstlgl`         | 0.628         | **Target**                                |
| `gdpcapitapct`    | 0.190         | ✓ Feature                                 |
| `ginicoefpct`     | −0.259        | ✓ Feature                                 |
| `econ_cluster`    | 0.242         | ✓ Feature                                 |
| `region_northern` | 0.268         | ✓ Feature                                 |
| `region_southern` | −0.136        | ✓ Feature                                 |
| `n_nonresp`       | −0.131        | ✓ Control                                 |
| `unemploymentpct` | 0.038         | ✗ Discard                                 |
| `inflationpct`    | −0.113        | ✗ Discard  (weak, and colineal with GDP)  |
| `lrscale`         | 0.058         | ✗ Discard                                 |
| `happy`           | 0.048         | ✗ Discard                                 |
| `agea`            | −0.050        | ✗ Discard                                 |
| `region_eastern`  | −0.106        | ✗ Redundant with econ_cluster (−0.655)    |
| `region_western`  | −0.073        | ✗ Null                                    |
"
------------------------------------------------------------------------------------------------------------